# NAVSIM Transfuser パイプライン

このノートブックでは、NAVSIM の Transfuser アーキテクチャ (timm ResNet-34 + GPT-style multi-scale fusion) を SageMaker AI ML Pipeline で学習・評価します。

### Transfuser とは

Transfuser は、前方カメラ画像と LiDAR の BEV (Bird's Eye View) ヒストグラムを CNN + Transformer で融合し、将来の軌跡を予測するセンサーエージェントです。[CARLA Garage](https://github.com/autonomousvision/carla_garage) の Transfuser バックボーンをベースにしています。

入力は以下の 3 つのモダリティです。

- **カメラ**: 前方 3 台 (cam_l0, cam_f0, cam_r0) をスティッチして 1024×256 の広角画像を生成
- **LiDAR**: 点群を 256×256 の BEV ヒストグラムに変換
- **EgoStatus**: 走行コマンド + 速度 + 加速度 (8 次元)

### Transfuser vs Latent Transfuser (LTF)

同じコンテナで 2 つのモードが動作します。切り替えはハイパーパラメータ `--latent` で行います。

| 項目 | Transfuser (`latent=false`) | LTF (`latent=true`) |
|------|---------------------------|--------------------|
| カメラ入力 | ✅ | ✅ |
| LiDAR 入力 | ✅ BEV ヒストグラム | ❌ 位置エンコーディングで代替 |
| 推論時の柔軟性 | LiDAR が必要 | LiDAR 不要 |

### EgoStatusMLP との違い

| 項目 | EgoStatusMLP | Transfuser |
|------|-------------|------------|
| 入力 | 速度・加速度・走行コマンド (8 次元) | カメラ画像 + LiDAR BEV + EgoStatus |
| モデル | 4 層 MLP (Multi-Layer Perceptron) | CNN + Transformer Encoder |
| GPU | 不要 | 必須 |
| データサイズ | 小さい (npz) | 大きい (pt) |
| 推奨インスタンス | `ml.c7i.xlarge` (CPU) | `ml.g6.4xlarge` (GPU) |

### コンテナ設計について

SageMaker で使用するコンテナ (Dockerfile) には navsim devkit をインストールしていません。navsim v1.1 は Python 3.9 + `numpy==1.23.4` 等の古いバージョンをピン留めしており、PyTorch DLC (Python 3.11) と依存関係が競合するためです。代わりに、特徴量抽出はこの notebook 上で事前に行い、学習・評価コンテナには前処理済みの pt データだけを渡す設計にしています。

### このノートブックの流れ

1. **パッケージインストール** - PyTorch、numpy 等の依存関係
2. **データセット準備** - ダミーデータ生成とデータの確認
3. **AWS セットアップ** - AWS リソース設定、S3 アップロード、コンテナビルド
4. **学習** - SageMaker Training Job で Transfuser を学習 (GPU)
5. **評価** - SageMaker Processing Job で ADE / FDE / PDM Score を計算
6. **結果確認** - 評価メトリクスの表示

---
## 1. パッケージインストール

In [ ]:
%pip install --quiet 'numpy<2' matplotlib torch

---
## 2. データセット準備

Transfuser の学習には、カメラ画像・LiDAR BEV・EgoStatus・GT 軌跡を含む前処理済みデータが必要です。

以下のセルを実行すると、パイプラインの動作確認用にダミーデータを生成して S3 にアップロードします。事前準備は不要で、そのまま実行してください。S3 に実データがアップロード済みの場合、既存データは上書きされません。

> **Tip**: 実際の走行データで学習したい場合は、JupyterLab ターミナルで `./pipelines/container-navsim-transfuser/scripts/prepare_dataset.sh` を実行すると OpenScene データセットから特徴量を抽出できます。

### データ形式

各サンプルは pt (PyTorch tensor) ファイルで、以下のキーを含みます。

| キー | Shape | 説明 |
|------|-------|------|
| `camera` | `[3, 256, 1024]` | スティッチ済みフロントカメラ画像 (RGB) |
| `lidar` | `[1, 256, 256]` | LiDAR BEV ヒストグラム |
| `status` | `[8]` | EgoStatus (速度・加速度・走行コマンド) |
| `trajectory` | `[8, 3]` | GT 軌跡 (x, y, heading) × 8 ポーズ |


In [ ]:
import os
import numpy as np
import torch

WORK_DIR = "/tmp/navsim_transfuser"
TRAIN_DIR = os.path.join(WORK_DIR, "train")
TEST_DIR = os.path.join(WORK_DIR, "test")
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

def generate_transfuser_dummy(data_dir, n_samples, num_poses=8):
    """パイプラインテスト用のダミーデータを生成する。"""
    for i in range(n_samples):
        camera = torch.randn(3, 256, 1024)  # RGB 画像
        lidar = torch.randn(1, 256, 256)    # LiDAR BEV
        status = torch.randn(8)              # EgoStatus
        trajectory = torch.zeros(num_poses, 3)
        for t in range(num_poses):
            trajectory[t, 0] = status[0] * (t + 1) * 0.5  # x = vx * t
            trajectory[t, 1] = status[1] * (t + 1) * 0.5  # y = vy * t
        trajectory += torch.randn_like(trajectory) * 0.1
        torch.save({
            "camera": camera, "lidar": lidar,
            "status": status, "trajectory": trajectory,
        }, os.path.join(data_dir, f"sample_{i:04d}.pt"))

# ダミーデータ生成 (train: 160, test: 40)
generate_transfuser_dummy(TRAIN_DIR, 160)
generate_transfuser_dummy(TEST_DIR, 40)
print(f"Train: {len(os.listdir(TRAIN_DIR))} samples")
print(f"Test:  {len(os.listdir(TEST_DIR))} samples")

### 2.1 データの確認

生成したダミーデータの中身を確認します。4 サンプルをカメラ画像・LiDAR BEV・軌跡のグリッドで表示します。

In [ ]:
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", message="TypedStorage is deprecated")

# データ構成の確認
sample = torch.load(os.path.join(TRAIN_DIR, "sample_0000.pt"), weights_only=True)
print("サンプルデータの構成:")
for k, v in sample.items():
    print(f"  {k:15s}: shape={list(v.shape)}, dtype={v.dtype}")
print()

# 4 サンプルをグリッド表示 (カメラ / LiDAR BEV / 軌跡)
sample_ids = [0, 10, 50, 100]
fig, axes = plt.subplots(len(sample_ids), 3, figsize=(18, 4 * len(sample_ids)))

for row, sid in enumerate(sample_ids):
    s = torch.load(os.path.join(TRAIN_DIR, f"sample_{sid:04d}.pt"), weights_only=True)

    # カメラ画像
    img = s["camera"].permute(1, 2, 0).detach().float()
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f"Sample #{sid} - Camera", fontsize=10)
    axes[row, 0].set_xlabel("width")
    axes[row, 0].set_ylabel("height")

    # LiDAR BEV
    axes[row, 1].imshow(s["lidar"][0].detach().float(), cmap="viridis")
    axes[row, 1].set_title(f"Sample #{sid} - LiDAR BEV", fontsize=10)
    axes[row, 1].set_xlabel("y")
    axes[row, 1].set_ylabel("x")

    # 軌跡
    traj = s["trajectory"].detach().tolist()
    traj_x = [0] + [p[0] for p in traj]
    traj_y = [0] + [p[1] for p in traj]
    speed = (s["status"][0]**2 + s["status"][1]**2)**0.5
    axes[row, 2].plot(traj_y, traj_x, "b-o", markersize=4, linewidth=1.5)
    axes[row, 2].plot(0, 0, "gs", markersize=8, label="Ego")
    axes[row, 2].plot(traj_y[-1], traj_x[-1], "r^", markersize=8, label="End")
    axes[row, 2].set_title(f"Sample #{sid} - Trajectory (speed={speed:.2f})", fontsize=10)
    axes[row, 2].set_xlabel("y (m)")
    axes[row, 2].set_ylabel("x (m, forward)")
    axes[row, 2].set_aspect("equal")
    axes[row, 2].grid(True, alpha=0.3)
    if row == 0:
        axes[row, 2].legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## 3. AWS セットアップ

AWS リソースの設定、S3 へのデータアップロード、コンテナイメージのビルドを行います。

CloudFormation スタック (`sagemaker-ai-ml-pipeline-stack`) から SageMaker 実行ロールの ARN を自動取得します。スタックが未作成の場合は、先に `infra/scripts/deploy.sh` を実行してください。

In [ ]:
%pip install --quiet "sagemaker>=2.200,<3" boto3

In [ ]:
import json
import logging
import boto3
logging.getLogger("sagemaker.jumpstart").setLevel(logging.CRITICAL)
import sagemaker
from sagemaker.estimator import Estimator
from sagemaker.pytorch.processing import PyTorchProcessor

PROJECT_NAME = os.environ.get("PROJECT_NAME", "sagemaker-ai-ml-pipeline")
REGION = boto3.session.Session().region_name or "us-east-1"
CONTAINER_TAG = "container-navsim-transfuser"

session = sagemaker.Session()
account_id = boto3.client("sts").get_caller_identity()["Account"]

cfn = boto3.client("cloudformation", region_name=REGION)
stack = cfn.describe_stacks(StackName=f"{PROJECT_NAME}-stack")["Stacks"][0]
outputs = {o["OutputKey"]: o["OutputValue"] for o in stack["Outputs"]}
role_arn = outputs["SageMakerRoleArn"]

# S3 パス
dataset_bucket = f"{PROJECT_NAME}-dataset-{account_id}-{REGION}"
model_bucket = f"{PROJECT_NAME}-model-{account_id}-{REGION}"
eval_bucket = f"{PROJECT_NAME}-eval-{account_id}-{REGION}"

train_data_uri = f"s3://{dataset_bucket}/{CONTAINER_TAG}/train"
test_data_uri = f"s3://{dataset_bucket}/{CONTAINER_TAG}/test"
model_output_uri = f"s3://{model_bucket}/output"
eval_output_uri = f"s3://{eval_bucket}/output"

ecr_image_uri = (
    f"{account_id}.dkr.ecr.{REGION}.amazonaws.com"
    f"/{PROJECT_NAME}-container:{CONTAINER_TAG}"
)

print(f"Project:       {PROJECT_NAME}")
print(f"Container:     {CONTAINER_TAG}")
print(f"Train data:    {train_data_uri}")
print(f"Test data:     {test_data_uri}")
print(f"ECR image:     {ecr_image_uri}")

### 3.1 S3 にアップロード

`prepare_dataset.sh` で S3 にデータがアップロード済みの場合はスキップされます。
Section 2 で生成したダミーデータのみアップロードします。


In [ ]:
s3 = boto3.client("s3", region_name=REGION)

# S3 に prepare_dataset.sh でアップロード済みのデータがあるか確認
existing = s3.list_objects_v2(
    Bucket=dataset_bucket, Prefix=f"{CONTAINER_TAG}/train/", MaxKeys=1,
)
if existing.get("KeyCount", 0) > 0:
    print(f"S3 に学習データが存在します: s3://{dataset_bucket}/{CONTAINER_TAG}/train/")
    print("アップロードをスキップします (prepare_dataset.sh のデータを使用)")
else:
    for split_name, split_dir in [(f"{CONTAINER_TAG}/train", TRAIN_DIR), (f"{CONTAINER_TAG}/test", TEST_DIR)]:
        for f in sorted(os.listdir(split_dir)):
            if not f.endswith(".pt"):
                continue
            s3.upload_file(
                os.path.join(split_dir, f),
                dataset_bucket,
                f"{split_name}/{f}",
            )
        print(f"Uploaded {split_name}/ ({len(os.listdir(split_dir))} files)")
    print("Upload complete.")


### 3.2 コンテナイメージのビルド & ECR プッシュ

SageMaker Training Job / Processing Job は ECR 上のコンテナイメージを使用します。初回実行時は以下のセルでビルド & プッシュを実行してください。

このコンテナは PyTorch DLC (GPU 版) をベースに、`timm` (画像バックボーン)、`opencv-python-headless` (画像前処理) 等を追加した軽量イメージです。navsim devkit はインストールしていません。

> **Note**: ECR にイメージが既にプッシュ済みの場合は、このセルをスキップできます。

In [ ]:
import subprocess

ecr_client = boto3.client("ecr", region_name=REGION)
image_exists = False
try:
    ecr_client.describe_images(
        repositoryName=f"{PROJECT_NAME}-container",
        imageIds=[{"imageTag": CONTAINER_TAG}],
    )
    image_exists = True
    print("ECR image already exists. Skipping build.")
    print(f"  {ecr_image_uri}")
except ecr_client.exceptions.ImageNotFoundException:
    print("ECR image not found. Building and pushing...")
except Exception as e:
    print(f"ECR check failed ({e}). Attempting build...")

if not image_exists:
    result = subprocess.run(
        ["bash", "../pipelines/scripts/02-build-and-push-container.sh",
         "-c", CONTAINER_TAG, "--auto-approve", PROJECT_NAME],
        text=True,
        env={**os.environ, "AWS_DEFAULT_REGION": REGION},
    )
    if result.returncode != 0:
        print(f"\nBuild failed (exit code {result.returncode}):")
    else:
        print(f"\nContainer pushed to {ecr_image_uri}")

---
## 4. 学習 (Training Job)

Transfuser モデルを SageMaker Training Job で学習します。

### 処理の流れ

1. SageMaker が S3 の学習データ (pt ファイル群) をコンテナの `SM_CHANNEL_TRAIN` にダウンロード
2. `train.py` が pt ファイルからカメラ画像・LiDAR BEV・EgoStatus・GT 軌跡を読み込み
3. CNN + Transformer で特徴量を融合し、L1 Loss で軌跡予測を学習
4. 学習済みモデル (`model.pth`) を `SM_MODEL_DIR` に保存

### インスタンスタイプについて

`ml.g6.4xlarge` (L4 1 台, 16 vCPU, 64 GiB) を使用します。CNN + Transformer の forward/backward が GPU バウンドなので、GPU が必須です。

### Transfuser / LTF の切り替え

下のセルの `USE_LATENT` を変更するだけで切り替えられます。

```python
USE_LATENT = "false"  # Transfuser (カメラ + LiDAR)
USE_LATENT = "true"   # Latent Transfuser (カメラのみ)
```

In [ ]:
from sagemaker.inputs import TrainingInput

# MLflow App ARN の取得 (存在する場合)
mlflow_app_arn = ""
mlflow_app_url = ""
try:
    cfn_client = boto3.client("cloudformation", region_name=REGION)
    resp = cfn_client.describe_stacks(StackName=f"{PROJECT_NAME}-stack")
    for output in resp["Stacks"][0]["Outputs"]:
        if output["OutputKey"] == "MlflowAppArn":
            mlflow_app_arn = output["OutputValue"]
            mlflow_app_url = f"https://{mlflow_app_arn.split('/')[-1]}.mlflow.sagemaker.{REGION}.amazonaws.com"
            break
    if mlflow_app_arn:
        print(f"MLflow App: {mlflow_app_arn}")
except Exception as e:
    print(f"MLflow App not found ({e}). Metrics will not be recorded.")

In [ ]:
# ============================================================
# Transfuser / LTF の切り替え
# ============================================================
# "false" → Transfuser (カメラ + LiDAR)
# "true"  → Latent Transfuser (カメラのみ)
USE_LATENT = "false"

estimator = Estimator(
    image_uri=ecr_image_uri,
    entry_point="train.py",
    source_dir=f"../pipelines/{CONTAINER_TAG}",
    role=role_arn,
    instance_count=1,
    instance_type="ml.g6.4xlarge",
    output_path=model_output_uri,
    base_job_name=f"{PROJECT_NAME}-navsim-transfuser-train",
    environment={
        "MLFLOW_APP_ARN": mlflow_app_arn,
        "MLFLOW_APP_URL": mlflow_app_url,
        "MODEL_GROUP_NAME": f"{PROJECT_NAME}-navsim-transfuser",
    },
    hyperparameters={
        "epochs": 30,
        "batch-size": 16,
        "learning-rate": 0.0003,
        "latent": USE_LATENT,
    },
    metric_definitions=[
        {"Name": "train:val_loss", "Regex": r"val_loss: ([0-9.]+)"},
        {"Name": "train:ade", "Regex": r"Training ADE: ([0-9.]+)"},
        {"Name": "train:fde", "Regex": r"Training FDE: ([0-9.]+)"},
    ],
    sagemaker_session=session,
)

mode = "Latent Transfuser (LTF)" if USE_LATENT == "true" else "Transfuser"
print(f"Mode: {mode}")
print("Estimator configured.")

### 学習ジョブの実行

`estimator.fit()` で SageMaker Training Job を起動します。GPU インスタンスの起動に数分かかります。

In [ ]:
estimator.fit(
    # input_mode: \"FastFile\" = S3 からオンデマンドストリーミング / \"File\" = 全量ダウンロード後に学習開始
    inputs={"train": TrainingInput(s3_data=train_data_uri, input_mode="FastFile")},
    wait=True,
    logs="All",
)

print(f"\nModel artifact: {estimator.model_data}")

---
## 5. 評価 (Processing Job)

学習済みモデルをテストデータで評価します。

### 評価メトリクス

| メトリクス | 説明 |
|-----------|------|
| `pdm_score` | ADE ベースの簡易総合スコア (0-1) |
| `ade` | Average Displacement Error。全タイムステップの平均 L2 距離 (m) |
| `fde` | Final Displacement Error。最終タイムステップの L2 距離 (m) |
| `heading_error` | heading の平均絶対誤差 (rad) |
| `miss_rate` | FDE > 2.0 m のサンプル割合 |

In [ ]:
eval_processor = PyTorchProcessor(
    framework_version="2.5.1",
    image_uri=ecr_image_uri,
    role=role_arn,
    instance_count=1,
    instance_type="ml.g6.4xlarge",
    base_job_name=f"{PROJECT_NAME}-navsim-transfuser-eval",
    env={
        "MLFLOW_APP_ARN": mlflow_app_arn,
        "MLFLOW_APP_URL": mlflow_app_url,
    },
    sagemaker_session=session,
)

eval_processor.run(
    inputs=[
        sagemaker.processing.ProcessingInput(
            source=estimator.model_data,
            destination="/opt/ml/processing/model",
        ),
        sagemaker.processing.ProcessingInput(
            source=test_data_uri,
            destination="/opt/ml/processing/test",
        ),
    ],
    outputs=[
        sagemaker.processing.ProcessingOutput(
            output_name="evaluation",
            source="/opt/ml/processing/evaluation",
            destination=eval_output_uri,
        ),
    ],
    code="evaluate.py",
    source_dir=f"../pipelines/{CONTAINER_TAG}",
)

print("Evaluation complete.")

---
## 6. 結果確認

評価結果 (`evaluation.json`) を S3 からダウンロードして表示します。

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as tmpdir:
    eval_key = "output/evaluation.json"
    local_eval = os.path.join(tmpdir, "evaluation.json")
    try:
        s3.download_file(eval_bucket, eval_key, local_eval)
        with open(local_eval) as f:
            metrics = json.load(f)
        mode = "LTF" if USE_LATENT == "true" else "Transfuser"
        print("=" * 50)
        print(f"NAVSIM {mode} 評価結果")
        print("=" * 50)
        for k, v in metrics.items():
            print(f"  {k:25s}: {v}")
        print("=" * 50)
    except Exception as e:
        print(f"Failed to download evaluation results: {e}")
        print(f"Check S3: s3://{eval_bucket}/output/")

---
## 7. Pipeline 実行 (オプション)

上記の学習・評価を SageMaker Pipeline として一括実行することもできます。

```bash
# ターミナルから実行 (データは事前に S3 にアップロード済みであること)
./pipelines/scripts/run-pipeline.sh -c container-navsim-transfuser --skip-upload
```

In [ ]:
# Pipeline 実行 (CLI 経由)
# !cd .. && ./pipelines/scripts/run-pipeline.sh -c {CONTAINER_TAG} --skip-upload